# Indoor temperature vs hour of day (MMA3001 project)

**Engineering question:** how well can a simple model predict room temperature from the hour of day?
A model like this is a baseline for HVAC control and for spotting unusual building behaviour.

**How to use this notebook:** run the cells from top to bottom. The first two cells *write* the helper code
(`temperature_tools.py`) and the tests (`test_temperature_tools.py`) into this folder, so commit those two files to GitHub as well.

In Colab, first run `!git clone https://github.com/USER/REPO` then `%cd REPO`.

## 0. Helper functions (written to temperature_tools.py)

In [ ]:
%%writefile temperature_tools.py
"""Helper functions for the indoor temperature project.

Everything the notebook does that can be tested on its own lives here.
Inputs are pandas DataFrames / NumPy arrays; temperatures are in degrees C,
times are Unix seconds (UTC) unless stated otherwise.

Typical use::

    raw = pd.read_csv("data.zip", compression="zip", usecols=["sensorid", "jsondata"])
    data = extract_temperatures(raw, sensor_id=6012002000869)
    data = clean_readings(data, "2024-05-01", "2025-01-01")
    data = select_good_days(data)
"""
import json

import numpy as np
import pandas as pd

# Columns the raw data must contain
REQUIRED_COLUMNS = ("sensorid", "jsondata")


def extract_temperatures(raw, sensor_id):
    """Pull the temperature readings for one sensor out of the raw JSON rows.

    Args:
        raw: DataFrame with columns "sensorid" and "jsondata" (a JSON string
            holding a list of variables, each with a list of timestamped values).
        sensor_id: Sensor ID to keep (same type as raw["sensorid"], e.g. int).

    Returns:
        DataFrame with columns "unix_time" and "temperature".
        Rows with unreadable JSON are skipped, not raised as errors.

    Raises:
        KeyError: If a required column is missing from raw.
        ValueError: If the sensor is not in the data, or has no temperature readings.
    """
    # Check the input shape before doing any work, so errors are clear
    missing = [c for c in REQUIRED_COLUMNS if c not in raw.columns]
    if missing:
        raise KeyError(f"Input is missing column(s): {missing}")
    available = sorted(raw["sensorid"].unique())
    if sensor_id not in available:
        raise ValueError(f"Sensor {sensor_id} not found. Available sensors: {available}")

    rows = []
    # Only this sensor's rows are parsed: JSON parsing is the slow part
    for text in raw.loc[raw["sensorid"] == sensor_id, "jsondata"]:
        try:
            variables = json.loads(text)
            for item in variables:
                # Each row also holds battery, humidity, CO2 etc.; keep temperature only
                if item["variable"]["name"] == "Temperature":
                    for reading in item["values"]:
                        rows.append((reading["time"], reading["value"]))
        except (ValueError, TypeError, KeyError):
            continue  # bad or unexpected row: skip it
    if not rows:
        raise ValueError(f"Sensor {sensor_id} has no readable temperature readings")
    return pd.DataFrame(rows, columns=["unix_time", "temperature"])


def clean_readings(data, start, end, timezone="Australia/Melbourne", valid_range=(-10, 50)):
    """Convert times, drop bad/duplicate readings and keep one date range.

    Args:
        data: DataFrame from extract_temperatures().
        start, end: Date strings; keeps start <= time < end (local time).
        timezone: Local time zone used for the "hour" and "date" columns.
        valid_range: (low, high) in degrees C. Readings outside this range are
            treated as sensor faults and dropped.

    Returns:
        New DataFrame sorted by time with extra columns
        "datetime", "date" and "hour" (0-23).

    Raises:
        ValueError: If no readings are left after cleaning.
    """
    data = data.copy()  # never change the caller's DataFrame
    # errors="coerce" turns text like "bad" into NaN so dropna() can remove it
    data["temperature"] = pd.to_numeric(data["temperature"], errors="coerce")
    data = data.dropna().drop_duplicates("unix_time")
    # Physically impossible indoor temperatures are sensor faults
    low, high = valid_range
    data = data[(data["temperature"] >= low) & (data["temperature"] <= high)]
    # Unix seconds are UTC; convert to local time so "hour" matches the clock on the wall
    data["datetime"] = pd.to_datetime(data["unix_time"], unit="s", utc=True).dt.tz_convert(timezone)
    data = data[(data["datetime"] >= start) & (data["datetime"] < end)]
    data = data.sort_values("datetime")
    if data.empty:
        raise ValueError("No readings left after cleaning (check dates, units and valid_range)")
    data["date"] = data["datetime"].dt.date
    data["hour"] = data["datetime"].dt.hour
    return data


def select_good_days(data, min_readings=120):
    """Keep only days with enough readings that also cover all 24 hours.

    Args:
        data: DataFrame from clean_readings() (needs "date", "hour", "temperature").
        min_readings: Minimum number of readings a day must have. At one reading
            every 10 minutes a full day has about 144.

    Returns:
        DataFrame containing only the readings from days that pass both checks.

    Raises:
        ValueError: If no day passes both checks.
    """
    # One row per day: how many readings, and how many different hours they cover
    per_day = data.groupby("date").agg(readings=("temperature", "size"),
                                       hours=("hour", "nunique"))
    good = per_day[(per_day["readings"] >= min_readings) & (per_day["hours"] == 24)].index
    if len(good) == 0:
        raise ValueError("No days have enough readings covering all 24 hours")
    return data[data["date"].isin(good)]


def split_days(days, train_frac=0.70, val_frac=0.15):
    """Split dates into train / validation / test in time order (no shuffling).

    Keeping time order means the test days come after the training days, like
    predicting the future from the past.

    Args:
        days: Iterable of dates.
        train_frac, val_frac: Fractions for train and validation; the rest is test.

    Returns:
        (train_days, val_days, test_days) as lists.

    Raises:
        ValueError: If there are fewer than 3 days (each split needs at least one).
    """
    days = sorted(days)
    n = len(days)
    if n < 3:
        raise ValueError("Need at least 3 days to make train, validation and test sets")
    a, b = int(n * train_frac), int(n * (train_frac + val_frac))  # cut positions
    return days[:a], days[a:b], days[b:]


def fit_piecewise(hours, hourly_avg, boundaries=(6, 8)):
    """Fit a straight line to each of three parts of the day.

    Parts are hours [0, b1], [b1, b2] and [b2, 23] (inclusive, so the boundary
    hours belong to two parts and the later part overwrites the earlier one).

    Args:
        hours: Array of hours 0..23.
        hourly_avg: Average temperature at each hour (same length as hours).
        boundaries: (b1, b2) hours where the lines change.

    Returns:
        Array of 24 predicted temperatures, one per hour.
    """
    hours = np.asarray(hours)
    hourly_avg = np.asarray(hourly_avg, dtype=float)
    b1, b2 = boundaries
    pred = np.zeros(len(hours))
    for start, end in [(0, b1), (b1, b2), (b2, 23)]:
        mask = (hours >= start) & (hours <= end)               # hours in this part
        coeffs = np.polyfit(hours[mask], hourly_avg[mask], 1)  # slope and intercept
        pred[mask] = np.polyval(coeffs, hours[mask])
    return pred


def rmse(actual, predicted):
    """Root mean square error: the typical size of a prediction error.

    Squaring means big misses count more than small ones.

    Args:
        actual: Measured values (array-like).
        predicted: Model predictions, same length as actual.

    Returns:
        RMSE as a float, in the same units as the inputs (degrees C here).
    """
    actual, predicted = np.asarray(actual, float), np.asarray(predicted, float)
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))


def mae(actual, predicted):
    """Mean absolute error: the average size of a prediction error.

    Args:
        actual: Measured values (array-like).
        predicted: Model predictions, same length as actual.

    Returns:
        MAE as a float, in the same units as the inputs (degrees C here).
    """
    actual, predicted = np.asarray(actual, float), np.asarray(predicted, float)
    return float(np.mean(np.abs(actual - predicted)))


def trapezoid_average(t_hours, values):
    """Time-weighted average of a signal using the trapezoidal rule.

    Unlike a simple mean, this accounts for uneven gaps between readings.

    Args:
        t_hours: Increasing times in hours (at least two points).
        values: Signal value at each time.

    Returns:
        Area under the curve divided by the total duration.

    Raises:
        ValueError: If there are fewer than two different times.
    """
    t_hours, values = np.asarray(t_hours, float), np.asarray(values, float)
    if len(t_hours) < 2 or t_hours[-1] == t_hours[0]:
        raise ValueError("Need at least two different times")
    area = np.trapezoid(values, t_hours)  # area under the curve (value x hours)
    return float(area / (t_hours[-1] - t_hours[0]))

## 0b. Tests (written to test_temperature_tools.py)

In [ ]:
%%writefile test_temperature_tools.py
"""Tests for temperature_tools. Run with:  pytest -v"""
import json

import numpy as np
import pandas as pd
import pytest

import temperature_tools as tt


def make_raw():
    """Tiny fake dataset: two sensors, one broken row."""
    good = json.dumps([{"variable": {"name": "Temperature"},
                        "values": [{"time": 1714521600, "value": 21.5}]},
                       {"variable": {"name": "Humidity"},
                        "values": [{"time": 1714521600, "value": 50}]}])
    other = json.dumps([{"variable": {"name": "Temperature"},
                         "values": [{"time": 1714521600, "value": 30.0}]}])
    return pd.DataFrame({"sensorid": [1, 2, 1],
                         "jsondata": [good, other, "not json at all"]})


def test_extract_keeps_only_requested_sensor_and_variable():
    """Only the chosen sensor's Temperature values come out (not humidity, not other sensors)."""
    out = tt.extract_temperatures(make_raw(), 1)
    assert len(out) == 1
    assert out["temperature"].iloc[0] == 21.5


def test_extract_skips_broken_json_without_crashing():
    """A row with unreadable JSON is skipped instead of crashing."""
    out = tt.extract_temperatures(make_raw(), 1)  # sensor 1 has a broken row
    assert list(out.columns) == ["unix_time", "temperature"]


def test_clean_removes_duplicates_and_missing():
    """Repeated timestamps and non-numeric values are removed."""
    raw = pd.DataFrame({"unix_time": [1714521600, 1714521600, 1714522200],
                        "temperature": [20.0, 20.0, "bad"]})
    out = tt.clean_readings(raw, "2024-05-01", "2024-06-01")
    assert len(out) == 1  # duplicate removed, "bad" -> NaN -> dropped


def test_clean_hour_uses_melbourne_time():
    """Hours are in local time: 00:00 UTC is 10:00 in Melbourne in May."""
    # 2024-05-01 00:00 UTC is 10:00 in Melbourne (UTC+10 in May)
    raw = pd.DataFrame({"unix_time": [1714521600], "temperature": [20.0]})
    out = tt.clean_readings(raw, "2024-05-01", "2024-06-01")
    assert out["hour"].iloc[0] == 10


def test_split_is_in_time_order_with_no_overlap():
    """Train, validation and test days are in time order and never overlap."""
    days = list(range(20))[::-1]  # deliberately unsorted
    train, val, test = tt.split_days(days)
    assert len(train) + len(val) + len(test) == 20
    assert max(train) < min(val) and max(val) < min(test)
    assert set(train).isdisjoint(val) and set(val).isdisjoint(test)


def test_rmse_and_mae_known_values():
    """RMSE and MAE match values worked out by hand."""
    actual = [1, 2, 3]
    predicted = [2, 2, 5]           # errors: 1, 0, 2
    assert tt.mae(actual, predicted) == pytest.approx(1.0)
    assert tt.rmse(actual, predicted) == pytest.approx(np.sqrt(5 / 3))


def test_trapezoid_of_constant_is_that_constant():
    """A constant signal must average to itself, even with uneven spacing."""
    t = np.array([0, 0.3, 1.0, 2.5])        # uneven spacing on purpose
    assert tt.trapezoid_average(t, np.full(4, 22.0)) == pytest.approx(22.0)


def test_trapezoid_of_straight_line_matches_analytical_answer():
    """Analytical check: the exact average of y = 2t + 1 on [0, 4] is 5."""
    # y = 2t + 1 on [0, 4]: exact average is the midpoint value 5
    t = np.array([0, 1.0, 1.5, 4.0])
    assert tt.trapezoid_average(t, 2 * t + 1) == pytest.approx(5.0)


def test_trapezoid_rejects_single_point():
    """One reading is not enough to integrate, so a ValueError is raised."""
    with pytest.raises(ValueError):
        tt.trapezoid_average([1.0], [20.0])


def test_piecewise_recovers_data_that_is_already_piecewise_linear():
    """If the data really is three straight lines, the fit should be (almost) exact."""
    hours = np.arange(24)
    # continuous curve with kinks exactly at hours 6 and 8 (slopes 0.1, 1.0, 0.5)
    truth = np.piecewise(
        hours.astype(float),
        [hours <= 6, (hours > 6) & (hours <= 8), hours > 8],
        [lambda h: 20.0 + 0.1 * h,
         lambda h: 20.6 + 1.0 * (h - 6),
         lambda h: 22.6 + 0.5 * (h - 8)],
    )
    pred = tt.fit_piecewise(hours, truth, boundaries=(6, 8))
    assert tt.rmse(truth, pred) < 1e-9


def test_piecewise_returns_one_value_per_hour():
    """The piecewise model gives exactly 24 predictions."""
    hours = np.arange(24)
    pred = tt.fit_piecewise(hours, np.linspace(20, 25, 24))
    assert pred.shape == (24,)


# ---- invalid / missing / unsupported inputs ----
def test_unknown_sensor_gives_clear_error():
    """Asking for a sensor that is not in the data raises a clear ValueError."""
    with pytest.raises(ValueError, match="not found"):
        tt.extract_temperatures(make_raw(), 999)


def test_missing_column_gives_clear_error():
    """Input without the required columns raises a KeyError."""
    with pytest.raises(KeyError):
        tt.extract_temperatures(pd.DataFrame({"sensorid": [1]}), 1)


def test_clean_drops_physically_impossible_temperatures():
    """A 99 C indoor reading is treated as a sensor fault and removed."""
    raw = pd.DataFrame({"unix_time": [1714521600, 1714522200],
                        "temperature": [21.0, 99.0]})   # 99 C indoors = sensor fault
    out = tt.clean_readings(raw, "2024-05-01", "2024-06-01")
    assert list(out["temperature"]) == [21.0]


def test_clean_raises_if_nothing_left():
    """A date range with no data raises a ValueError instead of returning nothing."""
    raw = pd.DataFrame({"unix_time": [1714521600], "temperature": [21.0]})
    with pytest.raises(ValueError):
        tt.clean_readings(raw, "2030-01-01", "2030-02-01")   # wrong date range


def test_select_good_days_raises_if_no_day_is_complete():
    """If no day has enough readings, a ValueError is raised."""
    raw = pd.DataFrame({"unix_time": [1714521600, 1714522200], "temperature": [21.0, 21.5]})
    data = tt.clean_readings(raw, "2024-05-01", "2024-06-01")
    with pytest.raises(ValueError):
        tt.select_good_days(data)


def test_split_needs_at_least_three_days():
    """Fewer than 3 days cannot make three splits, so a ValueError is raised."""
    with pytest.raises(ValueError):
        tt.split_days([1, 2])

## 1. Settings

In [ ]:
import io, json, time, timeit
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import temperature_tools as tt   # the helper file written above

# RAW link to the zip in your repo: swap USER and REPO for yours
# (check the branch name too: main or master)
ZIP_URL = "https://github.com/USER/REPO/raw/main/5EnvSensor_MayToDec2024_180kRows.zip"
LOCAL_ZIP = "5EnvSensor_MayToDec2024_180kRows.zip"   # used if the download fails

SENSOR_ID = 6012002000869
START, END = "2024-05-01", "2025-01-01"

## 2. Load the data
The zip is about 7 MB and holds a 350 MB csv. pandas unzips it for us.

In [ ]:
t_start = time.perf_counter()
try:
    r = requests.get(ZIP_URL, timeout=60)
    r.raise_for_status()
    if r.content[:2] != b"PK":      # real zip files start with "PK"
        raise ValueError("Downloaded file is not a zip (wrong link, or Git LFS pointer?)")
    source = io.BytesIO(r.content)
    print("Loaded from GitHub")
except Exception as e:
    print("GitHub download failed:", e)
    print("Using local file instead")
    source = LOCAL_ZIP

# only read the two columns we need (much faster and uses less memory)
raw = pd.read_csv(source, compression="zip", usecols=["sensorid", "jsondata"])
print(f"Rows: {len(raw)}   load time: {time.perf_counter() - t_start:.1f} s")

## 3. Clean one sensor's temperature readings

In [ ]:
# Filter by sensor FIRST, so we only parse that sensor's JSON.
data = tt.extract_temperatures(raw, SENSOR_ID)
data = tt.clean_readings(data, START, END)
print(len(data), "readings from", data["datetime"].min().date(), "to", data["datetime"].max().date())

## 4. Keep good days and split by date
Train 70% / validation 15% / test 15%, in time order (never shuffled, so the test days are genuinely in the future).

In [ ]:
data = tt.select_good_days(data, min_readings=120)
train_days, val_days, test_days = tt.split_days(data["date"].unique())
print(f"{len(train_days) + len(val_days) + len(test_days)} good days -> "
      f"train {len(train_days)}, validation {len(val_days)}, test {len(test_days)}")

train = data[data["date"].isin(train_days)]
val   = data[data["date"].isin(val_days)]
test  = data[data["date"].isin(test_days)]

## 5. Data quality check: how regular is the sampling?
The sensor should report every 10 minutes. Gaps are missing data the models never see.

In [ ]:
gaps = data.groupby("date")["datetime"].diff().dt.total_seconds().dropna() / 60   # minutes between readings
print(f"Median interval: {gaps.median():.2f} min   Mean: {gaps.mean():.2f} min   Max: {gaps.max():.2f} min")
print("Intervals longer than 15 minutes:", int((gaps > 15).sum()), "of", len(gaps))

## 6. Models (fitted on training data only)
Idea: average the temperature at each hour of the day (24 numbers), then describe those 24 numbers in different ways.

In [ ]:
hours = np.arange(24)
train_avg = train.groupby("hour")["temperature"].mean().to_numpy()

# Each model is a function that returns 24 predicted temperatures (one per hour).
curves = {
    "Hourly average (lookup)": lambda: train_avg.copy(),                              # no curve, just the 24 averages
    "Linear (baseline)":       lambda: np.polyval(np.polyfit(hours, train_avg, 1), hours),
    "Cubic":                   lambda: np.polyval(np.polyfit(hours, train_avg, 3), hours),
    "Piecewise linear":        lambda: tt.fit_piecewise(hours, train_avg, (6, 8)),
}
n_parameters = {"Hourly average (lookup)": 24, "Linear (baseline)": 2, "Cubic": 4, "Piecewise linear": 6}
interpretable = {
    "Hourly average (lookup)": "Very easy: it is just the data",
    "Linear (baseline)":       "Very easy: one slope",
    "Cubic":                   "Hard: coefficients have no physical meaning",
    "Piecewise linear":        "Easy: night, morning ramp, day slopes",
}
predictions = {name: make_curve() for name, make_curve in curves.items()}

## 7. Compare accuracy, runtime and complexity
Accuracy is measured on individual readings from days the models never saw.

In [ ]:
rows = []
for name, make_curve in curves.items():
    curve = predictions[name]
    seconds = min(timeit.repeat(make_curve, number=200, repeat=5)) / 200   # fit time per run
    rows.append({
        "Model": name,
        "Parameters": n_parameters[name],
        "Val RMSE (°C)":  tt.rmse(val["temperature"],  curve[val["hour"].to_numpy()]),
        "Test RMSE (°C)": tt.rmse(test["temperature"], curve[test["hour"].to_numpy()]),
        "Test MAE (°C)":  tt.mae(test["temperature"],  curve[test["hour"].to_numpy()]),
        "Fit time (ms)":  seconds * 1000,
        "Interpretability": interpretable[name],
    })

results = pd.DataFrame(rows).round(3)
results

## 8. Sensitivity: does the choice of piecewise boundaries matter?

In [ ]:
sens = []
for b1, b2 in [(5, 7), (5, 8), (6, 8), (6, 9), (7, 9)]:
    curve = tt.fit_piecewise(hours, train_avg, (b1, b2))
    sens.append([f"{b1:02d}:00", f"{b2:02d}:00", tt.rmse(val["temperature"], curve[val["hour"].to_numpy()])])

pd.DataFrame(sens, columns=["Boundary 1", "Boundary 2", "Validation RMSE (°C)"]).round(3)

## 9. Plot: unseen test days vs the models

In [ ]:
test_avg = test.groupby("hour")["temperature"].mean()

plt.figure(figsize=(10, 5))
plt.plot(test_avg.index, test_avg.values, "ko", label="Test data (hourly average)")
for name, curve in predictions.items():
    plt.plot(hours, curve, label=name)
plt.xlabel("Hour of day"); plt.ylabel("Temperature (°C)")
plt.title(f"Sensor {SENSOR_ID}: models vs unseen test days")
plt.xticks(range(0, 24, 2)); plt.grid(True); plt.legend()
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=200)
plt.show()

## 10. Numerical integration: daily time-weighted average (trapezoidal rule)
Checked against the simple mean. The two should be very close when readings are evenly spaced.

In [ ]:
daily = []
for date, day in data.groupby("date"):
    t_hours = (day["datetime"] - day["datetime"].iloc[0]).dt.total_seconds().to_numpy() / 3600
    daily.append([date, tt.trapezoid_average(t_hours, day["temperature"].to_numpy()), day["temperature"].mean()])

daily = pd.DataFrame(daily, columns=["Date", "Trapezoid average (°C)", "Simple mean (°C)"])
daily["Difference (°C)"] = daily["Trapezoid average (°C)"] - daily["Simple mean (°C)"]
print("Mean absolute difference:", round(daily["Difference (°C)"].abs().mean(), 4), "°C")
print("Max absolute difference: ", round(daily["Difference (°C)"].abs().max(), 4), "°C")

## 11. Performance: original loader vs improved loader
The original approach read every column, looped over every row with `iterrows()` and parsed every sensor's JSON.
The improved approach reads two columns and parses only the chosen sensor. This takes about 20 s;
set `RUN_BENCHMARK = False` to skip it.

In [ ]:
RUN_BENCHMARK = True

def original_loader(path):
    """Original method: all columns, iterrows, all sensors."""
    df = pd.read_csv(path, compression="zip")
    memory_mb = df.memory_usage(deep=True).sum() / 1e6
    records = []
    for _, row in df.iterrows():
        try:
            for item in json.loads(row["jsondata"]):
                if item["variable"]["name"] == "Temperature":
                    for reading in item["values"]:
                        records.append([row["sensorid"], reading["time"], reading["value"]])
        except (ValueError, TypeError, KeyError):
            continue
    return memory_mb

def improved_loader(path):
    """Improved method: two columns, one sensor."""
    df = pd.read_csv(path, compression="zip", usecols=["sensorid", "jsondata"])
    memory_mb = df.memory_usage(deep=True).sum() / 1e6
    tt.extract_temperatures(df, SENSOR_ID)
    return memory_mb

if RUN_BENCHMARK:
    bench = []
    for name, loader, repeats in [("Original", original_loader, 1), ("Improved", improved_loader, 3)]:
        times = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            memory_mb = loader(LOCAL_ZIP)
            times.append(time.perf_counter() - t0)
        bench.append([name, repeats, np.median(times), memory_mb])
    bench = pd.DataFrame(bench, columns=["Method", "Runs", "Median time (s)", "DataFrame memory (MB)"]).round(1)
    bench["Speed-up"] = (bench["Median time (s)"].iloc[0] / bench["Median time (s)"]).round(1)
    display(bench)

## 12. Profiler: where does the time actually go?
`cProfile` lists the functions that used the most time. Look at the top few lines.

In [ ]:
import cProfile, pstats

profiler = cProfile.Profile()
profiler.enable()
improved_loader(LOCAL_ZIP)
profiler.disable()

stats = pstats.Stats(profiler).strip_dirs().sort_stats("cumulative")
stats.print_stats(12);   # look for the read_csv and loads lines

## 13. Run the tests and write the test report

In [ ]:
!pip install -q pytest pytest-html
!python -m pytest -v --html=test_report.html --self-contained-html

## 14. Generate the HTML documentation from the docstrings

In [ ]:
!pip install -q pdoc
!python -m pdoc temperature_tools -o docs